- https://chatgpt.com/c/6a0956c8-7edc-8320-a707-9f4abfe58934
- trajectory-DPO 用这些偏序样本训练一个隐式 PRM；这个 PRM 通过 log-ratio 产出 step reward；step reward 再参与 policy model 的 RL 更新。
- iStar 让 policy 生成 rollouts，由 outcome verifier/model 排序形成正负轨迹对，再用 trajectory-based DPO 训练 implicit PRM，最后用 implicit step rewards 和 episode-level outcome rewards 一起更新 policy model。

### 偏序数据采集

- 先让当前 policy model 产生同一个任务 $x$ 下的一组轨迹
$$
\{\tau_1,\tau_2,\dots,\tau_N\},\quad \tau_i=(o^i_1,a^i_1,\dots,o^i_T,a^i_T).
$$
- 然后 outcome verifier/model 给每条完整轨迹一个结果分数：
$$
r_o(\tau_i).
$$
- 只要 $r_o(\tau_i)>r_o(\tau_j),$ 就构造一个偏好关系：$\tau_i \succ \tau_j.$
- $\mathcal D_{\text{pref}}=\{(x,\tau^+,\tau^-)\mid r_o(\tau^+)>r_o(\tau^-)\}.$
- WebShop 和 VisualSokoban 中 success rate 大于 0 的轨迹可作为 positive，SOTOPIA 中 goal completion score 大于 6 的轨迹可作为 positive。

### DPO

$$
J_{\text{PRM}}(\phi)
=-\mathbb E_{(\tau^+,\tau^-)}
\left[\log \sigma\left(\beta\log
\frac{\pi_\phi(\tau^+\mid x)}
{\pi_{\theta_{\text{old}}}(\tau^+\mid x)}
-\beta\log\frac{\pi_\phi(\tau^-\mid x)}{\pi_{\theta_{\text{old}}}(\tau^-\mid x)}\right)\right].
$$
- DPO loss 反传到 $\pi_\phi$, 也就是 implicit PRM，而 policy model 是 $\pi_\theta$。rajectory-DPO 目标写成上式。
    - $\pi_\theta \leftarrow \pi_{\theta_{\text{init}}},\qquad
\pi_{\theta_{\text{old}}}\leftarrow \pi_{\theta_{\text{init}}},\qquad
\pi_\phi \leftarrow \pi_{\theta_{\text{init}}}.$
    - policy model、old policy snapshot、implicit PRM 三者一开始来自同一个 base model。实验细节里也写到，implicit PRM initialized from the base policy model；例外是 VisualSokoban，policy 用 Qwen2.5-VL-7B-Instruct，PRM 用 Qwen2.5-7B-Instruct。
- DPO 的一个核心性质是：被训练模型相对 reference model 的 log-ratio 可以解释为 reward。本文的 implicit step reward 定义为：
$$
r_\phi(o_{1:t},a_t)
=\beta\log
\frac{\pi_\phi(a_t\mid o_{1:t},x)}{\pi_{\theta_{\text{old}}}(a_t\mid o_{1:t},x)}.
$$
- 这个 reward 没有来自显式 reward head，也没有来自人工 step label。它来自两个模型对同一个 action 的概率比值。若 $\pi_\phi$ 比旧 policy 更愿意给某个 action 高概率，则这个 action 的 implicit step reward 为正；反向则为负。本文也这样解释：该值衡量 freshly learned PRM 相对旧 policy 认为当前 action 更应被鼓励的程度。
- trajectory log-prob 可以按 step 分解（sigmoid/softmax 处理之前的统称为 logit）：
$$
\log \pi_\phi(\tau\mid x)=\sum_{t=1}^T\log \pi_\phi(a_t\mid o_{1:t},x).
$$
- 因此 DPO logit 也能分解为 step reward 的和：
$$
\beta\log
\frac{\pi_\phi(\tau^+\mid x)}
{\pi_{\theta_{\text{old}}}(\tau^+\mid x)}
-\beta\log
\frac{\pi_\phi(\tau^-\mid x)}
{\pi_{\theta_{\text{old}}}(\tau^-\mid x)}
=\sum_t r_\phi(o^+_{1:t},a^+_t)-\sum_t r_\phi(o^-_{1:t},a^-_t).
$$
- 把 trajectory-DPO 目标解释成 Bradley-Terry model 下的 step-wise reward function，并给出：
$$
P(\tau_1\succ\tau_2)
=\sigma\left(
\sum_t r_\phi(o^1_{1:t},a^1_t)
-\sum_t r_\phi(o^2_{1:t},a^2_t)
\right).
$$

### policy model 在哪里被训练

- policy model 没有直接用 trajectory-DPO loss 更新。本文后续用 implicit PRM 产出的 step reward 计算 step-level advantage：
$$
A^S(a^i_t)=\frac{r_\phi(a^i_t)-\text{mean}(R_s)}{\text{std}(R_s)}.
$$

### example code

- https://github.com/Tongyi-ConvAI/Qwen-Character/tree/main/CharacterRL-iStar